# Semantic plot embeddings

Compare pretrained semantic plot features with the lexical features from notebook 04. The model recipe remains fixed while the interaction evaluation uses the canonical **chronological 60/20/20 split within each user** created by notebook 01.

Run notebooks 01 and 04 first. Artifacts live under the split-specific processed directory; former global-cutoff results are deliberately not reused. Training users and validation users overlap by design, while individual rating events remain disjoint. Because different users have different time boundaries, this is an existing-user retrospective benchmark rather than a single historical deployment simulation.

All variants rank the same training movie catalog, exclude every movie the user already rated in training, and use the same positive threshold and Recall@10 denominator. Validation is used to compare the original and catalog-adjusted scoring hypotheses. The test labels are not loaded here.

## Load the canonical interaction split

Read the saved training and validation files and verify their raw-source hashes and split manifest. Reusing these files keeps this notebook consistent with the earlier baselines and avoids a second, divergent split implementation.

In [ ]:
from pathlib import Path
import hashlib
import json

import pandas as pd

SPLIT_ID = "per_user_chronological_60_20_20_v1"
POSITIVE_THRESHOLD = 4.0
TOP_K = 10

# Resolve the project from either its root or any nested notebook directory.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "raw" / "ml-latest-small").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Run from the project containing data/raw/ml-latest-small.")

raw_dir = data_dir = project_root / "data" / "raw" / "ml-latest-small"
processed_dir = project_root / "data" / "processed" / SPLIT_ID
manifest_path = processed_dir / "split_manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError("Run notebook 01 first to write the canonical split.")
split_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert split_manifest["split_id"] == SPLIT_ID
assert split_manifest["positive_threshold"] == POSITIVE_THRESHOLD
assert split_manifest["top_k"] == TOP_K

def file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for filename, expected_hash in split_manifest["raw_sha256"].items():
    if file_sha256(raw_dir / filename) != expected_hash:
        raise ValueError(f"{filename} changed; regenerate the split in notebook 01.")

movies = pd.read_csv(raw_dir / "movies.csv")
loaded_splits = {}
for name in ("train", "validation"):
    path = processed_dir / f"{name}_ratings.csv"
    expected = split_manifest["files"][path.name]
    if file_sha256(path) != expected["sha256"]:
        raise ValueError(f"{path.name} does not match the split manifest.")
    loaded_splits[name] = pd.read_csv(path, parse_dates=["rated_at"])
    assert len(loaded_splits[name]) == expected["rows"]

train_ratings = loaded_splits["train"]
validation_ratings = loaded_splits["validation"]

def save_validation_results(results, filename):
    """Attach the split and evaluation contract to each per-user result row."""
    saved = results.copy()
    saved["split_id"] = SPLIT_ID
    saved["split_fingerprint"] = split_manifest["split_fingerprint"]
    saved["evaluation_split"] = "validation"
    saved["positive_threshold"] = POSITIVE_THRESHOLD
    saved["k"] = TOP_K
    saved.to_csv(processed_dir / filename, index=False)

def validate_result_provenance(results, label="results"):
    expected = {
        "split_id": SPLIT_ID,
        "split_fingerprint": split_manifest["split_fingerprint"],
        "evaluation_split": "validation",
        "positive_threshold": POSITIVE_THRESHOLD,
        "k": TOP_K,
    }
    for column, value in expected.items():
        assert column in results and results[column].eq(value).all(), (
            f"{label}: missing or incompatible {column}; rerun its source notebook."
        )

print("Split:", SPLIT_ID)
print("Training ratings:", len(train_ratings))
print("Validation ratings:", len(validation_ratings))

## Load the fixed pretrained encoder

The sentence-transformer supplies plot features without fitting on MovieLens ratings. CPU execution preserves the original experiment and works without a GPU. Dependencies (`sentence-transformers`, `torch`, `numpy`, `pandas`, `scikit-learn`, and `tqdm`) should be installed in the project environment before running; this notebook does not modify the environment. The initial model download requires internet access unless cached.

In [ ]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device="cpu"
)

print(
    "Embedding dimensions:",
    embedding_model.get_sentence_embedding_dimension()
)

print(
    "Maximum input length:",
    embedding_model.max_seq_length
)

## Sanity-check semantic similarity

These short synthetic sentences illustrate the encoder output and cosine similarity before applying it to movie plots. This is an implementation illustration, not evidence that personalized recommendations will improve.

In [ ]:
example_texts = [
    "A detective investigates a mysterious disappearance.",
    "An investigator tries to find a missing person.",
    "A chef opens a restaurant and prepares dinner."
]

example_embeddings = embedding_model.encode(
    example_texts,
    normalize_embeddings=True
)

print("Embedding matrix shape:", example_embeddings.shape)

example_similarities = pd.DataFrame(
    cosine_similarity(example_embeddings),
    index=["Detective", "Missing person", "Chef"],
    columns=["Detective", "Missing person", "Chef"]
)

example_similarities

## Load the training candidate plot catalog

Notebook 04 writes one row per training movie, including movies without matched plots. Validate its split fingerprint, file hash, and movie IDs before using it. Feature matrices contain only nonblank plots in sorted movie-ID order; the recommendation candidate set still contains every training movie.

In [ ]:
catalog_manifest_path = processed_dir / "plot_catalog_manifest.json"
if not catalog_manifest_path.is_file():
    raise FileNotFoundError("Run notebook 04 to regenerate the plot catalogs.")
plot_catalog_manifest = json.loads(catalog_manifest_path.read_text(encoding="utf-8"))
assert plot_catalog_manifest["split_id"] == SPLIT_ID
assert plot_catalog_manifest["split_fingerprint"] == split_manifest["split_fingerprint"]
catalog_path = processed_dir / "training_plot_catalog.csv"
assert file_sha256(catalog_path) == plot_catalog_manifest["files"][catalog_path.name]["sha256"]

training_plot_catalog = pd.read_csv(
    catalog_path, dtype={"wikipedia_movie_id": "Int64"}
)
assert training_plot_catalog["movieId"].is_unique
assert set(training_plot_catalog["movieId"]) == set(train_ratings["movieId"])
training_plot_catalog["has_plot"] = (
    training_plot_catalog["plot"].fillna("").str.strip().ne("")
)
text_movies = (
    training_plot_catalog.loc[training_plot_catalog["has_plot"]]
    .sort_values("movieId").reset_index(drop=True).copy()
)
assert not text_movies.empty, "No training movies have usable plot text."
print("Training candidates:", len(training_plot_catalog))
print("Movies with plots:", len(text_movies))
print("Candidate plot coverage:", f"{training_plot_catalog['has_plot'].mean():.1%}")

## Measure plot lengths before choosing chunks

Token counts include model special tokens. Compare them with the model input limit to quantify how much text direct, truncated encoding would discard. Counts are generated from the current training catalog rather than copied from an earlier split.

In [ ]:
def count_plot_tokens(plot):
    encoded = embedding_model.tokenizer(
        plot,
        add_special_tokens=True,
        truncation=False,
        verbose=False
    )
    return len(encoded["input_ids"])


text_movies["token_count"] = text_movies["plot"].map(
    count_plot_tokens
)

input_limit = embedding_model.max_seq_length

exceeds_limit = text_movies["token_count"] > input_limit

print("Model input limit:", input_limit)
print("Plots exceeding the limit:", exceeds_limit.sum())
print("Percentage exceeding the limit:", f"{exceeds_limit.mean():.1%}")

text_movies["token_count"].describe()

## Reserve room for special tokens

Subtract the tokenizer special-token count from the input limit. The resulting content limit is used for nonoverlapping chunks so no plot content is silently truncated.

In [ ]:
tokenizer = embedding_model.tokenizer

special_token_count = tokenizer.num_special_tokens_to_add(pair=False)

content_token_limit = (
    embedding_model.max_seq_length - special_token_count
)

print("Special tokens per chunk:", special_token_count)
print("Plot tokens allowed per chunk:", content_token_limit)

## Create nonoverlapping token chunks

Tokenize each plot once without special tokens, then retain every content token in order. Store the originating movie ID, chunk number, and content length for reproducible aggregation and row-alignment checks.

In [ ]:
chunk_rows = []

for movie in text_movies.itertuples(index=False):
    token_ids = tokenizer(
        movie.plot,
        add_special_tokens=False,
        truncation=False,
        verbose=False
    )["input_ids"]

    for chunk_number, start in enumerate(
        range(0, len(token_ids), content_token_limit),
        start=1
    ):
        chunk_token_ids = token_ids[
            start:start + content_token_limit
        ]

        chunk_rows.append({
            "movieId": movie.movieId,
            "chunk_number": chunk_number,
            "content_token_count": len(chunk_token_ids),
            "token_ids": chunk_token_ids
        })

plot_chunks = pd.DataFrame(chunk_rows)

## Verify chunk coverage

Each chunk must fit the content limit, and its movie-level content lengths must sum to the original full token count. These checks distinguish intentional chunking from accidental truncation or duplicate text.

In [ ]:
assert (
    plot_chunks["content_token_count"]
    .between(1, content_token_limit)
    .all()
)

tokens_per_movie = plot_chunks.groupby("movieId")[
    "content_token_count"
].sum()

expected_tokens = (
    text_movies.set_index("movieId")["token_count"]
    - special_token_count
)

assert tokens_per_movie.reindex(
    expected_tokens.index
).eq(expected_tokens).all()

chunks_per_movie = plot_chunks.groupby("movieId").size()

print("Movies represented:", len(chunks_per_movie))
print("Total chunks:", len(plot_chunks))
print("Fewest chunks per movie:", chunks_per_movie.min())
print("Most chunks per movie:", chunks_per_movie.max())

display(
    plot_chunks[
        ["movieId", "chunk_number", "content_token_count"]
    ].head(8)
)

## Inspect one padded model batch

This encoder uses BERT-style CLS and SEP tokens. Add them to each chunk, pad the batch, and mark nonpadding positions with the attention mask. Padding is excluded by the model pooling implementation; special tokens retain the pretrained model’s normal treatment.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

small_chunks = plot_chunks.head(4)

batch_token_ids = [
    [tokenizer.cls_token_id]
    + token_ids
    + [tokenizer.sep_token_id]
    for token_ids in small_chunks["token_ids"]
]

batch_length = max(len(ids) for ids in batch_token_ids)

assert batch_length <= embedding_model.max_seq_length

input_ids = torch.full(
    (len(batch_token_ids), batch_length),
    fill_value=tokenizer.pad_token_id,
    dtype=torch.long
)

attention_mask = torch.zeros_like(input_ids)

for row_number, token_ids in enumerate(batch_token_ids):
    length = len(token_ids)

    input_ids[row_number, :length] = torch.tensor(
        token_ids,
        dtype=torch.long
    )
    attention_mask[row_number, :length] = 1

batch_inputs = {
    "input_ids": input_ids,
    "attention_mask": attention_mask,
    "token_type_ids": torch.zeros_like(input_ids)
}

print("Input matrix shape:", tuple(input_ids.shape))
print(
    "Non-padding tokens per chunk:",
    attention_mask.sum(dim=1).tolist()
)

## Run an inference-only batch

Disable training behavior and gradient tracking. Normalize the model’s sentence embeddings to unit length so their dot product can later be interpreted as cosine similarity.

In [ ]:
batch_inputs = {
    name: tensor.to(embedding_model.device)
    for name, tensor in batch_inputs.items()
}

embedding_model.eval()

with torch.inference_mode():
    model_output = embedding_model(batch_inputs)

    normalized_embeddings = F.normalize(
        model_output["sentence_embedding"],
        p=2,
        dim=1
    )

    small_chunk_embeddings = normalized_embeddings.cpu().numpy()

print("Embedding matrix shape:", small_chunk_embeddings.shape)

print(
    "Embedding L2 norms:",
    np.linalg.norm(small_chunk_embeddings, axis=1)
)

print(
    "All values finite:",
    np.isfinite(small_chunk_embeddings).all()
)

## Encapsulate the chunk encoding recipe

The reusable function implements the same tensor construction shown above. It keeps the pretrained model pooling and normalizes each chunk vector; it does not train or fine-tune the encoder on interaction labels.

In [ ]:
def embed_chunk_batch(token_id_lists, model):
    """Embed token chunks using our BERT-style model."""
    batch_tokenizer = model.tokenizer

    sequences = [
        [batch_tokenizer.cls_token_id]
        + token_ids
        + [batch_tokenizer.sep_token_id]
        for token_ids in token_id_lists
    ]

    batch_length = max(len(ids) for ids in sequences)

    assert batch_length <= model.max_seq_length

    input_ids = torch.full(
        (len(sequences), batch_length),
        fill_value=batch_tokenizer.pad_token_id,
        dtype=torch.long,
        device=model.device
    )

    attention_mask = torch.zeros_like(input_ids)

    for row_number, token_ids in enumerate(sequences):
        length = len(token_ids)

        input_ids[row_number, :length] = torch.tensor(
            token_ids,
            dtype=torch.long,
            device=model.device
        )
        attention_mask[row_number, :length] = 1

    features = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "token_type_ids": torch.zeros_like(input_ids)
    }

    model.eval()

    with torch.inference_mode():
        output = model(features)

        embeddings = F.normalize(
            output["sentence_embedding"],
            p=2,
            dim=1
        )

        return embeddings.cpu().numpy()

## Encode all training plot chunks

Process the chunks in bounded batches and preserve their row order in the output array. Batch size affects memory and throughput, not the token coverage or intended scoring rule.

In [ ]:
from tqdm import tqdm

batch_size = 16
embedding_dimensions = embedding_model.get_sentence_embedding_dimension()

chunk_embeddings = np.empty(
    (len(plot_chunks), embedding_dimensions),
    dtype=np.float32
)

for start in tqdm(
    range(0, len(plot_chunks), batch_size),
    desc="Embedding plot chunks",
    unit="batch"
):
    stop = min(start + batch_size, len(plot_chunks))

    token_id_lists = (
        plot_chunks["token_ids"]
        .iloc[start:stop]
        .tolist()
    )

    chunk_embeddings[start:stop] = embed_chunk_batch(
        token_id_lists,
        embedding_model
    )

## Validate and persist chunk vectors

Check finite values and unit norms before saving. The archive includes its movie/chunk identifiers and split fingerprint so a cached array cannot be mistaken for features from the former global-date split.

In [ ]:
chunk_norms = np.linalg.norm(chunk_embeddings, axis=1)

assert np.isfinite(chunk_embeddings).all()
assert np.allclose(chunk_norms, 1.0, atol=1e-5)

print("Chunk embedding shape:", chunk_embeddings.shape)
print("Smallest L2 norm:", chunk_norms.min())
print("Largest L2 norm:", chunk_norms.max())

np.savez_compressed(
    processed_dir / "plot_chunk_embeddings.npz",
    embeddings=chunk_embeddings,
    split_id=np.asarray(SPLIT_ID),
    split_fingerprint=np.asarray(split_manifest["split_fingerprint"]),
    movie_ids=plot_chunks["movieId"].to_numpy(),
    chunk_numbers=plot_chunks["chunk_number"].to_numpy(),
    content_token_counts=plot_chunks["content_token_count"].to_numpy()
)

print("Saved plot_chunk_embeddings.npz")

## Aggregate chunks into one vector per movie

Average a movie’s unit chunk vectors using content-token counts as weights, then normalize the result. A short final chunk contributes less than a full chunk. Explicit movie-ID alignment prevents positional mismatches between vectors and catalog rows.

In [ ]:
# Positions of each movie's chunks in chunk_embeddings
chunk_positions = plot_chunks.groupby("movieId", sort=False).indices
chunk_lengths = plot_chunks["content_token_count"].to_numpy()

assert len(plot_chunks) == len(chunk_embeddings)
assert text_movies["movieId"].is_unique

movie_embeddings = np.empty(
    (len(text_movies), chunk_embeddings.shape[1]),
    dtype=np.float32
)

# Preserve the exact row order of text_movies
for movie_row, movie_id in enumerate(text_movies["movieId"]):
    positions = chunk_positions[movie_id]

    movie_embeddings[movie_row] = np.average(
        chunk_embeddings[positions],
        axis=0,
        weights=chunk_lengths[positions]
    )

# Check the averages before normalizing
movie_norms = np.linalg.norm(movie_embeddings, axis=1)

assert np.isfinite(movie_embeddings).all()
assert (movie_norms > 0).all()

# Give every movie vector unit length
movie_embeddings /= movie_norms[:, None]

## Persist movie vectors with provenance

Save the movie IDs alongside the embedding array and record the model recipe, source catalog hash, split fingerprint, and archive hashes. Any later reuse must match those IDs and provenance. These are features derived from training-catalog plots, with no validation ratings used in their construction.

In [ ]:
final_norms = np.linalg.norm(movie_embeddings, axis=1)

assert np.isfinite(movie_embeddings).all()
assert np.allclose(final_norms, 1.0, atol=1e-5)

print("Movie embedding shape:", movie_embeddings.shape)
print("Smallest L2 norm:", final_norms.min())
print("Largest L2 norm:", final_norms.max())

np.savez_compressed(
    processed_dir / "movie_plot_embeddings.npz",
    embeddings=movie_embeddings,
    split_id=np.asarray(SPLIT_ID),
    split_fingerprint=np.asarray(split_manifest["split_fingerprint"]),
    movie_ids=text_movies["movieId"].to_numpy()
)

print("Saved movie_plot_embeddings.npz")
embedding_manifest = {
    "split_id": SPLIT_ID,
    "split_fingerprint": split_manifest["split_fingerprint"],
    "training_catalog_sha256": file_sha256(catalog_path),
    "model_name": EMBEDDING_MODEL_NAME,
    "model_revision": getattr(embedding_model[0].auto_model.config, "_commit_hash", None),
    "max_seq_length": int(embedding_model.max_seq_length),
    "chunking": "nonoverlapping full content tokens; reserve tokenizer special tokens",
    "chunk_pooling": "pretrained model sentence_embedding; L2 normalize",
    "movie_pooling": "content-token-weighted chunk mean; L2 normalize",
    "movie_ids": text_movies["movieId"].astype(int).tolist(),
    "files": {
        name: {"sha256": file_sha256(processed_dir / name)}
        for name in ("plot_chunk_embeddings.npz", "movie_plot_embeddings.npz")
    },
}
(processed_dir / "embedding_manifest.json").write_text(
    json.dumps(embedding_manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

## Inspect movie-to-movie neighbors

For unit movie vectors, a dot product is cosine similarity. The Toy Story example checks self-similarity and inspects the ranked neighbors. Its qualitative result must be interpreted separately from held-out recommendation accuracy.

In [ ]:
# Prefer Toy Story when available; otherwise inspect the first plot movie.
seed_movie_id = 1 if text_movies["movieId"].eq(1).any() else int(text_movies["movieId"].iloc[0])
toy_story_position = np.flatnonzero(
    text_movies["movieId"].to_numpy() == seed_movie_id
)[0]

# Compare Toy Story with every movie
similarities = (
    movie_embeddings @ movie_embeddings[toy_story_position]
)

print("Similarity scores calculated:", len(similarities))
print(
    "Seed movie similarity to itself:",
    similarities[toy_story_position]
)

similarity_results = text_movies[["movieId", "title"]].copy()
similarity_results["similarity"] = similarities

# Exclude Toy Story itself and rank the remaining movies
ranked_neighbors = (
    similarity_results
    .loc[similarity_results["movieId"] != seed_movie_id]
    .sort_values(
        ["similarity", "movieId"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

display(ranked_neighbors.head(10))

## Inspect the same comparison titles as TF-IDF

Show selected related and unrelated titles when they are present in this training plot catalog. Compare rankings and aggregate validation performance across feature representations; raw similarity magnitudes have different distributions.

In [ ]:
comparison_movie_ids = [
    3114,  # Toy Story 2
    2041,  # Condorman
    318    # The Shawshank Redemption
]

display(
    ranked_neighbors.loc[
        ranked_neighbors["movieId"].isin(comparison_movie_ids)
    ]
)

## Define the embedding recommender

Build an unnormalized mean vector from the user’s positively rated training movies with plot features. Its dot product with each unit movie vector equals mean cosine similarity. Exclude all previously rated training movies, including low ratings. Sort by score, training rating count, and movie ID; users with no usable plot likes receive the popularity fallback. The optional catalog mean subtracts each movie’s general similarity baseline. Missing-plot candidates receive score zero, which can place them above negative embedding scores; that is the retained experiment policy.

In [ ]:
def recommend_embedding_based(
    user_id,
    training_data,
    text_movies,
    movie_embeddings,
    candidate_catalog,
    n=TOP_K,
    catalog_mean_vector=None
):
    """Rank fitting-catalog candidates by mean plot similarity or its adjustment."""
    history = training_data.loc[
        training_data["userId"] == user_id
    ]

    liked_ids = history.loc[
        history["rating"] >= POSITIVE_THRESHOLD,
        "movieId"
    ]

    liked_mask = text_movies["movieId"].isin(liked_ids).to_numpy()

    recommendations = candidate_catalog.loc[
        ~candidate_catalog["movieId"].isin(history["movieId"]),
        ["movieId", "title", "has_plot"]
    ].copy()

    recommendations["score"] = 0.0

    if liked_mask.any():
        # Average the vectors of the user's liked movies
        user_profile = movie_embeddings[liked_mask].mean(axis=0)
        
        if catalog_mean_vector is not None:
            user_profile = user_profile - catalog_mean_vector
        # Mean cosine similarity, optionally minus the catalog baseline
        movie_scores = movie_embeddings @ user_profile

        scores_by_id = pd.Series(
            movie_scores,
            index=text_movies["movieId"]
        )

        recommendations["score"] = (
            recommendations["movieId"]
            .map(scores_by_id)
            .fillna(0.0)
        )

    training_counts = training_data["movieId"].value_counts()

    recommendations["training_rating_count"] = (
        recommendations["movieId"]
        .map(training_counts)
        .fillna(0)
        .astype(int)
    )

    return (
        recommendations
        .sort_values(
            ["score", "training_rating_count", "movieId"],
            ascending=[False, False, True]
        )
        .head(n)
        .reset_index(drop=True)
    )

## Inspect personalized recommendations

Display example users without assuming either is a cold-start user. The per-user split supplies training history for every user, although an individual user can still lack positive ratings or usable plot likes.

In [ ]:
for user_id in [1, 2]:
    print(f"\nRecommendations for user: {user_id}")

    display(
        recommend_embedding_based(
            user_id=user_id,
            training_data=train_ratings,
            text_movies=text_movies,
            movie_embeddings=movie_embeddings,
            candidate_catalog=training_plot_catalog,
            n=10
        )
    )

## Evaluate the original embeddings on validation

Evaluate every user with at least one validation rating at or above the shared positive threshold. Recall@10 keeps all held-out positive movies in its denominator, including movies unseen in training or without plot features. Users with no positive validation ratings have undefined recall and are omitted. The mean weights each evaluated user equally. Assertions check candidate membership, uniqueness, and seen-movie exclusions.

In [ ]:
validation_positives = validation_ratings.loc[
    validation_ratings["rating"] >= POSITIVE_THRESHOLD
]

training_users = set(train_ratings["userId"])
evaluation_rows = []

for user_id, user_positives in validation_positives.groupby("userId"):
    recommendations = recommend_embedding_based(
        user_id=user_id,
        training_data=train_ratings,
        text_movies=text_movies,
        movie_embeddings=movie_embeddings,
        candidate_catalog=training_plot_catalog,
        n=TOP_K
    )

    seen_ids = set(train_ratings.loc[train_ratings["userId"] == user_id, "movieId"])
    recommended_ids_check = set(recommendations["movieId"])
    assert recommendations["movieId"].is_unique
    assert len(recommendations) <= TOP_K
    assert recommended_ids_check.isdisjoint(seen_ids)
    assert recommended_ids_check.issubset(set(train_ratings["movieId"]))
    liked_plot_ids = set(train_ratings.loc[
        (train_ratings["userId"] == user_id)
        & (train_ratings["rating"] >= POSITIVE_THRESHOLD), "movieId"
    ]).intersection(set(text_movies["movieId"]))

    positive_movie_ids = set(user_positives["movieId"])
    recommended_movie_ids = set(recommendations["movieId"])

    hits = len(
        positive_movie_ids.intersection(recommended_movie_ids)
    )

    evaluation_rows.append({
        "userId": user_id,
        "positive_movie_count": len(positive_movie_ids),
        "hits_at_10": hits,
        "recall_at_10": hits / len(positive_movie_ids),
        "has_training_history": user_id in training_users,
        "has_liked_training_plot": bool(liked_plot_ids)
    })

embedding_results = pd.DataFrame(evaluation_rows)

print("Users evaluated:", len(embedding_results))
print(
    "Embedding mean Recall@10:",
    embedding_results["recall_at_10"].mean()
)

display(
    embedding_results
    .groupby("has_liked_training_plot")
    .agg(
        users=("userId", "size"),
        mean_recall_at_10=("recall_at_10", "mean")
    )
)

## Measure recommendation concentration

Count how often each candidate is recommended across evaluated users. A broad mean profile may repeatedly favor movies similar to many plots. Concentration is a diagnostic to investigate, not a conclusion about the new split before execution.

In [ ]:
warm_results = embedding_results.copy()

print("Evaluated users:", len(warm_results))
print(
    "Users with at least one hit:",
    warm_results["hits_at_10"].gt(0).sum()
)
print("Total hits:", warm_results["hits_at_10"].sum())

recommendation_tables = []

for user_id in warm_results["userId"]:
    recommendations = recommend_embedding_based(
        user_id=user_id,
        training_data=train_ratings,
        text_movies=text_movies,
        movie_embeddings=movie_embeddings,
        candidate_catalog=training_plot_catalog,
        n=10
    )

    recommendations["userId"] = user_id
    recommendation_tables.append(recommendations)

warm_recommendations = (
    pd.concat(recommendation_tables, ignore_index=True)
    if recommendation_tables else
    pd.DataFrame(columns=["movieId", "title", "userId"])
)

recommendation_frequency = (
    warm_recommendations
    .groupby(["movieId", "title"], as_index=False)
    .agg(
        users_recommended_to=("userId", "nunique")
    )
    .sort_values(
        ["users_recommended_to", "movieId"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

print(
    "Distinct movies across these recommendation lists:",
    warm_recommendations["movieId"].nunique()
)

display(recommendation_frequency.head(15))

## Compare repetition with catalog-wide similarity

The average unit movie vector is the training plot catalog mean. A movie’s dot product with that mean equals its average cosine similarity to the catalog. Join this diagnostic to recommendation frequency to inspect whether generally similar movies repeatedly rise in the rankings.

In [ ]:
# Average vector across all movies with plot features
catalog_mean_vector = movie_embeddings.mean(axis=0)

# Each movie's average cosine similarity to the plot catalog
catalog_mean_similarities = (
    movie_embeddings @ catalog_mean_vector
)

catalog_similarity = text_movies[["movieId", "title"]].copy()

catalog_similarity["mean_catalog_similarity"] = (
    catalog_mean_similarities
)

catalog_similarity = (
    catalog_similarity
    .sort_values(
        ["mean_catalog_similarity", "movieId"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

catalog_similarity["catalog_similarity_rank"] = (
    np.arange(1, len(catalog_similarity) + 1)
)

# Compare general catalog similarity with recommendation frequency
frequency_diagnostic = recommendation_frequency.merge(
    catalog_similarity[
        [
            "movieId",
            "mean_catalog_similarity",
            "catalog_similarity_rank"
        ]
    ],
    on="movieId",
    how="left",
    validate="one_to_one"
)

display(
    frequency_diagnostic
    .sort_values(
        ["users_recommended_to", "movieId"],
        ascending=[False, True]
    )
    .head(15)
)

## Evaluate a catalog-baseline adjustment

Subtract the training catalog mean from each user profile before scoring. This is equivalent to subtracting each candidate’s average catalog similarity from its original user-specific score. The adjustment uses only training features. Validation evaluates the hypothesis; the test set remains reserved.

In [ ]:
catalog_mean_vector = movie_embeddings.mean(axis=0)

adjusted_rows = []

for user_id, user_positives in validation_positives.groupby("userId"):
    recommendations = recommend_embedding_based(
        user_id=user_id,
        training_data=train_ratings,
        text_movies=text_movies,
        movie_embeddings=movie_embeddings,
        candidate_catalog=training_plot_catalog,
        n=TOP_K,
        catalog_mean_vector=catalog_mean_vector
    )

    seen_ids = set(train_ratings.loc[train_ratings["userId"] == user_id, "movieId"])
    recommended_ids_check = set(recommendations["movieId"])
    assert recommendations["movieId"].is_unique
    assert len(recommendations) <= TOP_K
    assert recommended_ids_check.isdisjoint(seen_ids)
    assert recommended_ids_check.issubset(set(train_ratings["movieId"]))
    liked_plot_ids = set(train_ratings.loc[
        (train_ratings["userId"] == user_id)
        & (train_ratings["rating"] >= POSITIVE_THRESHOLD), "movieId"
    ]).intersection(set(text_movies["movieId"]))

    positive_ids = set(user_positives["movieId"])
    recommended_ids = set(recommendations["movieId"])

    hits = len(positive_ids.intersection(recommended_ids))

    adjusted_rows.append({
        "userId": user_id,
        "positive_movie_count": len(positive_ids),
        "hits_at_10": hits,
        "recall_at_10": hits / len(positive_ids),
        "has_training_history": user_id in training_users,
        "has_liked_training_plot": bool(liked_plot_ids)
    })

adjusted_embedding_results = pd.DataFrame(adjusted_rows)

print("Users evaluated:", len(adjusted_embedding_results))

print(
    "Original embedding Recall@10:",
    embedding_results["recall_at_10"].mean()
)

print(
    "Adjusted embedding Recall@10:",
    adjusted_embedding_results["recall_at_10"].mean()
)

display(
    adjusted_embedding_results
    .groupby("has_liked_training_plot")
    .agg(
        users=("userId", "size"),
        mean_recall_at_10=("recall_at_10", "mean")
    )
)

## Compare both variants user by user

Join the evaluations on user ID and verify matching recall denominators before calculating paired changes. Report improved, worsened, and unchanged users rather than inferring the distribution of gains from an aggregate mean.

In [ ]:
comparison_columns = [
    "userId",
    "positive_movie_count",
    "hits_at_10",
    "recall_at_10"
]

embedding_comparison = (
    embedding_results.loc[
        embedding_results["has_training_history"],
        comparison_columns
    ]
    .merge(
        adjusted_embedding_results[comparison_columns],
        on="userId",
        how="left",
        suffixes=("_original", "_adjusted"),
        validate="one_to_one"
    )
)

# Confirm both evaluations used the same recall denominators
assert (
    embedding_comparison["positive_movie_count_original"]
    == embedding_comparison["positive_movie_count_adjusted"]
).all()

embedding_comparison["recall_change"] = (
    embedding_comparison["recall_at_10_adjusted"]
    - embedding_comparison["recall_at_10_original"]
)

change = embedding_comparison["recall_change"]

print("Users compared:", len(embedding_comparison))
print("Improved:", change.gt(0).sum())
print("Worsened:", change.lt(0).sum())
print("Unchanged:", change.eq(0).sum())

print(
    "Total hits before:",
    embedding_comparison["hits_at_10_original"].sum()
)
print(
    "Total hits after:",
    embedding_comparison["hits_at_10_adjusted"].sum()
)

display(
    embedding_comparison.loc[
        change.ne(0),
        [
            "userId",
            "hits_at_10_original",
            "hits_at_10_adjusted",
            "recall_at_10_original",
            "recall_at_10_adjusted",
            "recall_change"
        ]
    ]
    .sort_values("recall_change", ascending=False)
    .reset_index(drop=True)
)

## Save comparable validation artifacts

Persist both per-user evaluations and the paired diagnostic under the canonical split directory. The shared save helper stamps each result with the split fingerprint and evaluation settings. Notebook 06 checks that provenance before comparing models.

In [ ]:
for results in (embedding_results, adjusted_embedding_results, embedding_comparison):
    results["plot_catalog_sha256"] = file_sha256(catalog_path)

save_validation_results(embedding_results, "embedding_validation_results.csv")
save_validation_results(
    adjusted_embedding_results, "adjusted_embedding_validation_results.csv"
)
save_validation_results(embedding_comparison, "embedding_adjustment_comparison.csv")
print("Saved embedding validation results and paired comparison.")

## Reusable embedding recipe for final fitting

The helper below packages the same demonstrated chunk-and-aggregate procedure for a later fitting catalog, such as training plus validation. It returns vectors in the supplied catalog row order and does not consume ratings. Keeping this recipe explicit prevents the final evaluation from silently switching to direct truncated full-plot embeddings.

In [ ]:
"""Reusable version of notebook05's fixed plot embedding recipe."""
import numpy as np
import pandas as pd


def embed_chunk_batch(token_id_lists, model):
    """Embed BERT-style chunks with model pooling and unit-length outputs."""
    import torch
    import torch.nn.functional as F

    tokenizer = model.tokenizer
    if not token_id_lists:
        return np.empty((0, model.get_sentence_embedding_dimension()), dtype=np.float32)
    sequences = [
        [tokenizer.cls_token_id] + ids + [tokenizer.sep_token_id]
        for ids in token_id_lists
    ]
    batch_length = max(map(len, sequences))
    if batch_length > model.max_seq_length:
        raise ValueError("A chunk exceeds the model's input limit.")
    input_ids = torch.full(
        (len(sequences), batch_length), tokenizer.pad_token_id,
        dtype=torch.long, device=model.device,
    )
    attention_mask = torch.zeros_like(input_ids)
    for row, ids in enumerate(sequences):
        input_ids[row, :len(ids)] = torch.tensor(ids, dtype=torch.long, device=model.device)
        attention_mask[row, :len(ids)] = 1
    features = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "token_type_ids": torch.zeros_like(input_ids),
    }
    model.eval()
    with torch.inference_mode():
        output = model(features)
        return F.normalize(output["sentence_embedding"], p=2, dim=1).cpu().numpy()


def embed_movie_plots(catalog, model, batch_size=16):
    """Return unit vectors in catalog row order, preserving all plot tokens.

    The caller supplies unique movieId rows with nonblank plot strings.
    Chunk vectors use the pretrained model's normal pooling; aggregation
    weights are content-token counts. No interaction labels are consumed.
    """
    if batch_size < 1:
        raise ValueError("batch_size must be positive.")
    if not catalog["movieId"].is_unique:
        raise ValueError("catalog movieId values must be unique.")
    if catalog["plot"].fillna("").str.strip().eq("").any():
        raise ValueError("Only movies with nonblank plots can be embedded.")
    dimensions = model.get_sentence_embedding_dimension()
    if catalog.empty:
        return np.empty((0, dimensions), dtype=np.float32)
    tokenizer = model.tokenizer
    content_limit = model.max_seq_length - tokenizer.num_special_tokens_to_add(pair=False)
    if content_limit < 1:
        raise ValueError("The input limit leaves no room for content tokens.")
    chunks = []
    owners = []
    lengths = []
    for movie_row, plot in enumerate(catalog["plot"]):
        ids = tokenizer(
            plot, add_special_tokens=False, truncation=False, verbose=False
        )["input_ids"]
        if not ids:
            raise ValueError(f"Plot at catalog row {movie_row} has no content tokens.")
        for start in range(0, len(ids), content_limit):
            chunk = ids[start:start + content_limit]
            chunks.append(chunk)
            owners.append(movie_row)
            lengths.append(len(chunk))
    chunk_vectors = np.empty((len(chunks), dimensions), dtype=np.float32)
    for start in range(0, len(chunks), batch_size):
        stop = min(start + batch_size, len(chunks))
        chunk_vectors[start:stop] = embed_chunk_batch(chunks[start:stop], model)
    owners = np.asarray(owners)
    lengths = np.asarray(lengths)
    movie_vectors = np.empty((len(catalog), dimensions), dtype=np.float32)
    for movie_row in range(len(catalog)):
        positions = np.flatnonzero(owners == movie_row)
        movie_vectors[movie_row] = np.average(
            chunk_vectors[positions], axis=0, weights=lengths[positions]
        )
    norms = np.linalg.norm(movie_vectors, axis=1)
    if not np.isfinite(movie_vectors).all() or not (norms > 0).all():
        raise ValueError("Movie vector aggregation produced nonfinite or zero vectors.")
    movie_vectors /= norms[:, None]
    return movie_vectors

## Interpretation and review boundaries

The generated tables provide the current-split findings. Do not carry numerical conclusions from the former global-date split into this experiment. Qualitative movie neighbors, recommendation concentration, and mean validation recall answer different questions; inspect all three before attributing a change to semantic understanding.

Incomplete plot coverage, conservative title/year linkage, pretrained encoder assumptions, and averaging diverse user preferences can limit the model. A zero score for missing plot features is an explicit fallback policy, particularly relevant when embedding scores are negative. External plot text and encoder training data are not constrained to each user's historical rating date, so this remains a retrospective offline study.

Both embedding variants are candidates for the later validation model comparison. Any selection must be made from validation results before refitting on training plus validation and evaluating the reserved test portion.